In [12]:
# from google.colab import drive
# drive.mount('/content/drive')

In [13]:
# # !pip install py7zr
# from google.colab import drive
# import py7zr
# import os

# # Define the path to the .7z file
# seven_zip_file_path = '/content/drive/MyDrive/datasets/dataset.7z'

# # Define the directory where you want to extract the files
# extract_to_dir = '/content/drive/MyDrive/datasets'

# # Make sure the extraction directory exists
# os.makedirs(extract_to_dir, exist_ok=True)

# # Extract the .7z file
# with py7zr.SevenZipFile(seven_zip_file_path, mode='r') as z:
#     z.extractall(path=extract_to_dir)

# print(f"Extracted files to {extract_to_dir}")

In [4]:
import tensorflow as tf

# Define the path to your directory (it should contain subfolders for each class)
dataset_directory = 'dataset'

# Load the dataset from the directory
training = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_directory,
    image_size=(224, 224),  # Resize images to a standard size (optional)
    batch_size=32,  # Adjust the batch size according to your needs
    label_mode='binary', # Can be 'int' (default), 'categorical', or 'binary'
    seed=123,
    validation_split=0.4,
    subset='training'
)

testing = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_directory,
    image_size=(224, 224),  # Resize images to a standard size (optional)
    batch_size=32,  # Adjust the batch size according to your needs
    label_mode='binary', # Can be 'int' (default), 'categorical', or 'binary'
    seed=123,
    validation_split=0.4,
    subset='validation'
)

# Add rescaling to normalize pixel values to the range [0, 1]
rescale = tf.keras.layers.Rescaling(1. / 255)

# # Apply rescaling to the dataset
# testing = testing.map(lambda x, y: (rescale(x), y))


# Explore the dataset
for images, labels in training.take(1):
    print(images.shape)  # Shape of the batch (32, 256, 256, 3) for example
    print(labels.shape)  # Shape of the labels   


Found 132 files belonging to 2 classes.
Using 80 files for training.
Found 132 files belonging to 2 classes.
Using 52 files for validation.
(32, 224, 224, 3)
(32, 1)


In [7]:
training.class_names

['image_bullying', 'non_bullying_images']

In [15]:
for images, labels in training.take(1):
    print(images)

tf.Tensor(
[[[[ 15.886161    33.88616     37.88616   ]
   [ 15.341517    32.316963    35.975445  ]
   [ 16.569197    32.138393    33.430805  ]
   ...
   [183.86157    180.         139.43079   ]
   [185.65848    180.65848    140.65848   ]
   [187.77228    182.77228    142.77228   ]]

  [[ 15.886161    33.88616     37.88616   ]
   [ 15.341517    32.316963    35.975445  ]
   [ 16.569197    32.138393    33.430805  ]
   ...
   [179.52014    175.65857    135.08936   ]
   [184.20758    179.20758    139.20758   ]
   [185.67854    180.67854    140.67854   ]]

  [[ 15.886161    33.88616     37.88616   ]
   [ 15.341517    32.316963    35.975445  ]
   [ 16.569197    32.138393    33.430805  ]
   ...
   [172.92029    169.05872    128.02219   ]
   [179.3192     174.3192     133.23438   ]
   [180.54686    175.54686    134.46204   ]]

  ...

  [[ 17.65626     32.656258    39.656258  ]
   [ 19.381727    34.38173     40.698692  ]
   [ 22.707588    37.70759     42.70759   ]
   ...
   [  0.70299137  48.890

In [ ]:
# Optional: Prefetch for performance
testing = testing.prefetch(buffer_size=tf.data.AUTOTUNE)
training = training.prefetch(buffer_size=tf.data.AUTOTUNE)

In [16]:
import tensorflow as tf

# Define a more extensive data augmentation pipeline
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),  # Randomly flip both horizontally and vertically
    tf.keras.layers.RandomRotation(0.3),  # Randomly rotate by up to 30% of 360 degrees
    tf.keras.layers.RandomZoom(height_factor=(-0.2, 0.2), width_factor=(-0.2, 0.2)),  # Randomly zoom in and out
    tf.keras.layers.RandomTranslation(height_factor=0.2, width_factor=0.2),  # Translate images up to 20% of height/width
    tf.keras.layers.GaussianNoise(0.1),  # Add random Gaussian noise with a standard deviation of 0.1
])

augmented_train_dataset = training.map(lambda x, y: (data_augmentation(x, training=True), y))
augmented_train_dataset = augmented_train_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)


In [17]:
for image, labels in testing.take(1):
  print(image.shape)

(26, 224, 224, 3)


In [18]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Flatten, Dense, Dropout, Rescaling
from tensorflow.keras.optimizers import Adam

base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze the base model

model = tf.keras.Sequential([
    base_model,
    Rescaling(1./255),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_3 (Rescaling)         │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │     1,605,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,320,449 (62.26 MB)

 Trainable params: 1,605,761 (6.13 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [19]:
# import os
# from PIL import Image

# directory = "dataset"

# for folder in os.listdir(directory):
#     folder_path = os.path.join(directory, folder)
#     if os.path.isdir(folder_path):
#         for file in os.listdir(folder_path):
#             file_path = os.path.join(folder_path, file)
#             try:
#                 img = Image.open(file_path)
#                 img.verify()  # Verify the image file
#             except (IOError, SyntaxError):
#                 print(f"Invalid or unsupported image file: {file_path}")
#                 os.remove(file_path)  # Optionally remove the problematic file

In [20]:
# from PIL import Image

# source_directory = "dataset"
# target_format = "png"  # Change to 'jpeg' or 'png' as needed

# for folder in os.listdir(source_directory):
#     folder_path = os.path.join(source_directory, folder)
#     if os.path.isdir(folder_path):
#         for file in os.listdir(folder_path):
#             file_path = os.path.join(folder_path, file)
#             try:
#                 img = Image.open(file_path)
#                 new_file_path = os.path.splitext(file_path)[0] + f".{target_format}"
#                 img.save(new_file_path)
#                 print(f"Converted {file} to {target_format}")
#                 os.remove(file_path)  # Remove the old file
#             except Exception as e:
#                 print(f"Failed to convert {file}: {e}")


In [21]:
from tensorflow.keras.callbacks import EarlyStopping 

early = EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    restore_best_weights=True
)

from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,  # Reduce LR by a factor of 0.2
    patience=5,  # Wait for 3 epochs before reducing LR
    min_lr=1e-6  # Minimum learning rate
)

In [22]:
# model.fit(
#     augmented_train_dataset,  # Training dataset
#     validation_data=testing,  # Validation dataset
#     epochs=50,  # Number of epochs
#     callbacks=[early, reduce_lr]
#  )

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 12s 3s/step - accuracy: 0.6695 - loss: 0.6223 - val_accuracy: 0.5000 - val_loss: 1.1272 - learning_rate: 0.0010
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.7757 - loss: 0.5776 - val_accuracy: 0.5000 - val_loss: 1.1385 - learning_rate: 0.0010
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.8111 - loss: 0.4216 - val_accuracy: 0.5000 - val_loss: 0.9689 - learning_rate: 0.0010
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.7945 - loss: 0.4197 - val_accuracy: 0.5000 - val_loss: 0.7795 - learning_rate: 0.0010
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.7937 - loss: 0.3922 - val_accuracy: 0.5385 - val_loss: 0.7328 - learning_rate: 0.0010
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.8152 - loss: 0.3997 - val_accuracy: 0.5385 - val_loss: 0.7769 - learning_rate: 0.0010
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.7909 - loss: 0.3454 - val_accuracy: 0.5769 - val_loss

In [28]:
model.predict(testing)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


array([[7.6319110e-01],
       [1.4981219e-01],
       [4.0627692e-05],
       [1.1724289e-04],
       [5.8819771e-02],
       [6.9245748e-04],
       [8.1580019e-01],
       [1.1831163e-03],
       [2.0528443e-01],
       [8.8686300e-03],
       [8.5747898e-01],
       [5.7914163e-05],
       [8.0100429e-01],
       [7.6391888e-01],
       [8.5925555e-01],
       [6.5063214e-05],
       [2.0571816e-01],
       [3.7586279e-04],
       [7.2187190e-03],
       [6.3583541e-01],
       [2.5161544e-03],
       [2.4370043e-02],
       [7.5373960e-01],
       [8.7794733e-01],
       [1.4656083e-01],
       [8.6155039e-01]], dtype=float32)

In [24]:
model.save('sense_media.keras')
print('file saved succesfully')

file saved succesfully
